[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap07/cap07.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 **Practical Part with Programming Exercises**

This list of programming exercises (EP) consolidates the theoretical formulations presented throughout Chapter 7 — Image Classification and Pattern Recognition — through an applied practical track. Unlike the direct pixel manipulation of the previous chapters, the EPs in this chapter work with the **intermediate quantities** of a real pattern recognition *pipeline* — feature vectors, distances, predicted and actual labels, local binary patterns, and orientation histograms — allowing each step of the reasoning to be manually validated without relying on external machine learning libraries.

The sequencing of the exercises reproduces the conceptual flow of the chapter: it begins with the manual implementation of the **k-NN** classifier decision rule over a small feature space; then, from the perspective of feature normalization, the classifier implemented in the first exercise of the list is revisited; it proceeds to the calculation of the **evaluation** metrics (confusion matrix, precision, and recall) from predicted and actual labels; it continues with the manual coding of the **LBP** texture descriptor from a $3\times3$ neighborhood; it delves into the computation of the orientation histogram of the **HOG** descriptor for a single cell; it then advances to the integration of **descriptor extraction**, **k-NN classification**, and **multi-class evaluation** into a complete texture recognition *pipeline*; and it concludes with the application of this same *pipeline* to a **real image** (PGM format), in which the LBP descriptor is computed directly on the pixels of a texture mosaic.

### 🎯 Objective of this Notebook

This notebook allows you to develop, validate, organize, and test solutions for **Programming Exercises (EPs)** in interactive environments, such as Colab, using the same test cases as Moodle, and copying them there only when registering the official grade.

### *Download*

Download `morph.py` and `testsuite.py` by running the cell below:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Running the Tests
To evaluate the tests, run `TestSuite("EP07_01.extension").run()` in a new cell, replacing the extension with that of the language used (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system downloads the test cases from GitHub, runs the program, and calculates the grade automatically.

To test Python code directly, without saving a file, use `run_code(code)` passing the code as a *string* in a `code` variable:

```python
code = """
# ... your code here ...
"""
TestSuite("EP07_01").run_code(code)
```

### 🛠️ Summary of the `morph.py` Methods (Ch. 7)

The `morph.py` library provides two versions for most algorithms: a **pedagogical** one (methods ending in `0`), implemented step by step in NumPy, and a **classical** one, based on the `scikit-learn` and `scikit-image` libraries. The pedagogical implementations are used in the **Programming Exercises (PEs)**, as they do not depend on external libraries and run within the memory limits of the Moodle **VPL** environment. The classical versions, in turn, are more efficient and suitable for experiments in environments such as Colab and Jupyter Notebook, but they usually **cannot be used in the Moodle PEs**, as the `scikit-learn` library exceeds the memory available in the VPL.

1. **Data reading (`readClasses`, `readDataset`, `readTrain`, `readTest`)**  
   They standardize the input of the training and test sets, returning the feature matrices ($X$) and the label vectors ($y$).

2. **Classification (`knn0` / `knn`)**  
   They implement the **k-nearest neighbors (k-NN)** algorithm for binary and multiclass classification, using Euclidean or Manhattan distance.

3. **Normalization (`zscore0` / `zscore`)**  
   They apply *z-score* normalization to the attributes, reducing scale differences before classification.

4. **Evaluation (`confusion0` / `confusion`)**  
   They compute the confusion matrix and metrics such as accuracy, precision, and recall, for both binary and multiclass problems.

5. **Texture descriptor (`lbp0` / `lbp`)**  
   They compute the ***Local Binary Pattern* (LBP)**, allowing one to obtain the LBP map, the code of a single pixel, or the histogram of an image region.

6. **Shape descriptor (`hog0` / `hog`)**  
   They compute the ***Histogram of Oriented Gradients* (HOG)**, producing histograms of gradient orientations to represent shape and contour information.

### EP07_01 🟢 Step-by-Step k-NN Classifier

The `KNeighborsClassifier` from `scikit-learn`, used throughout the chapter, hides behind a single call (`.fit` / `.predict`) a quite simple decision rule: for each new observation, compute the distance to all training examples, select the $k$ closest ones, and vote by the majority class among them.

Before relying on the library, you have been tasked with implementing this rule from scratch, for a two-dimensional feature space, exactly as the chapter's interactive decision boundary simulator does internally with each user click.

#### 📋 Implementation Guidelines

1. **Quantity and parameter:** Read the integer $N$ (number of training examples) and the odd integer $k$ (number of neighbors).
2. **Training examples:** For each of the $N$ examples, read three values: the $x$ and $y$ coordinates (real numbers) and the label $r$ (integer, $0$ or $1$).
3. **Queries:** Read the integer $Q$ (number of query points) and then the coordinates $x_q$, $y_q$ (real numbers) of each query.
4. **Distance:** For each query, compute the Euclidean distance to **all** training examples:
$$
d(x_q, x_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}.
$$
5. **Neighbor selection:** Sort the examples by increasing distance and select the first $k$. In case of a **distance tie** at the boundary of the $k$-th neighbor, break the tie by the example read **first** in the input (stable reading order).
6. **Majority voting:** Count the votes for each class among the $k$ selected neighbors. If there is a **tie in the voting** (only possible when $k$ is even, which should not occur per the guideline in item 1, but handle defensively), assign the class of the closest neighbor among the tied classes.
7. **Output:** For each query, in input order, print the predicted class. At the end, print the total number of queries classified as class `1`.

#### 📌 Computational Constraints

* **Fixed metric:** use exclusively the Euclidean distance (not the *squared distance*) for the sorting, although the comparison result is the same.
* **$k$ always odd:** the input guarantees $k$ odd and $k \le N$; still, implement the tie-breaking from item 6 for robustness.
* **Stability:** when sorting by distance, preserve the relative order of examples with the same distance (stable sorting).

#### 🧠 Theoretical Foundation

| Element | Role in k-NN |
|---|---|
| Feature space | Set of all possible vectors $(x, y)$ |
| Euclidean distance | Measure of similarity between observations |
| Small $k$ | Irregular boundary, high variance |
| Large $k$ | Smooth boundary, high bias |
| Majority voting | Decision rule $\hat y = \operatorname{mode}\{y_i : x_i \in N_k(x)\}$ |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $N$ and $k$, separated by spaces.
* Next $N$ lines: three values per line — $x$, $y$ (real numbers) and $r$ (integer $\in \{0,1\}$), separated by spaces.
* Next line: integer $Q$.
* Next $Q$ lines: two values per line — $x_q$, $y_q$ (real numbers), separated by spaces.

**Output:**

* $Q$ lines, each with the predicted class (`0` or `1`) for the respective query, in input order.
* Last line: `Total class 1: X`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4 3<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>1<br>1 1 | 0<br>Total class 1: 0 | Query close to the class 0 cluster. |
| 4 1<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>2<br>0.9 0.1<br>5.5 5.1 | 0<br>1<br>Total class 1: 1 | With $k=1$, each query inherits the class of its closest neighbor. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0701" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0701 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0701 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0701 button:hover { background: #e8dfcf; }
  #sim-ep0701 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0701_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0701_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP07_01: k-NN Classifier Step by Step</span>
  <span class="sim-ep0701_pill">Majority Voting</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0701_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Number of Neighbors (k): <span id="sim-ep0701_vl" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    
    <input id="sim-ep0701_sl" type="range" min="1" max="7" step="2" value="3">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust k and see which training examples (ordered by distance) participate in the voting for the fixed query (&starf; at x = 3, y = 3).
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0701_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0701_debug" class="sim-ep0701_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep01(root){
    if (!root || root.dataset.sim07Ep01Init) return;
    root.dataset.sim07Ep01Init = "1";

    var query = {x: 3, y: 3};
    var pontos = [
      {nome: "A", x: 0, y: 0, r: 0},
      {nome: "B", x: 1, y: 0, r: 0},
      {nome: "C", x: 5, y: 5, r: 1},
      {nome: "D", x: 6, y: 5, r: 1},
      {nome: "E", x: 2, y: 2, r: 0},
      {nome: "F", x: 4, y: 4, r: 1},
      {nome: "G", x: 0, y: 2, r: 0},
      {nome: "H", x: 6, y: 3, r: 1}
    ];

    pontos.forEach(function(p, i){
      p.d = Math.sqrt(Math.pow(p.x - query.x, 2) + Math.pow(p.y - query.y, 2));
      p.idx = i;
    });

    pontos.sort(function(a, b){
      return (a.d - b.d) || (a.idx - b.idx);
    });

    var slEl  = root.querySelector('#sim-ep0701_sl');
    var vlEl  = root.querySelector('#sim-ep0701_vl');
    var cards = root.querySelector('#sim-ep0701_cards');
    var dbg   = root.querySelector('#sim-ep0701_debug');

    function render(){
      var k = parseInt(slEl.value, 10);
      vlEl.textContent = k;
      cards.innerHTML = '';
      var votos = [0, 0];

      pontos.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">d = ' + p.d.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : pontos[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] + '  |  Classe prevista: ' + previsto;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim07Ep01(){
    var root = document.getElementById('sim-ep0701');
    if (root) initSim07Ep01(root); else setTimeout(tryInitSim07Ep01, 200);
  }
  tryInitSim07Ep01();
})();
</script>
""")

**Figura 7.1:** EP07_01 Simulator: Step-by-Step k-NN Classifier


<figure id="fig-07-sim-ep0701">
  <img src="imagens/fig-07-sim-ep0701.png" alt=" EP07_01 Simulator: Step-by-Step k-NN Classifier " style="max-width:80%" />
  <figcaption><strong>Figura 7.1:</strong>  EP07_01 Simulator: Step-by-Step k-NN Classifier </figcaption>
</figure>

In [ ]:
%%writefile EP07_01.py
# Python code

In [ ]:
TestSuite("EP07_01.py").run()

### EP07_02 🟡 *Z-score* Normalization and Robustness of k-NN to Distinct Scales

This exercise revisits the classifier implemented in **EP07_01**, this time under the perspective discussed in the section *The Impact of Scale and Feature Normalization* of the chapter: k-NN decides based on the distance between vectors, so that a feature measured on a much larger scale than the others tends to **dominate** the distance calculation, even when it is not the most relevant for separating the classes.

An inspection system records, for each part, its **area** (in pixels, possibly reaching hundreds or thousands) and its **circularity** (always between $0$ and $1$). You have been tasked with classifying new parts by k-NN in two ways — with and without the *Z-score* standardization presented in the chapter — and reporting in which cases the two approaches **diverge**.

#### 📋 Implementation Guidelines

1. **Quantity and parameter:** Read the integer $N$ (number of training examples) and the odd integer $k$.
2. **Training examples:** For each of the $N$ examples, read three values: the area $x_1$ (real), the circularity $x_2$ (real), and the label $r$ (integer, $0$ or $1$).
3. **Queries:** Read the integer $Q$ and then the coordinates $x_1, x_2$ of each query.
4. **Classification without normalization:** For each query, classify it by k-NN directly on $(x_1, x_2)$, with Euclidean distance and the same tie-breaking rules as in EP07_01 (reading order for tied distances; nearest neighbor among tied classes in the majority vote).
5. **Normalization parameters:** Compute the mean $\mu_j$ and the **population** standard deviation $\sigma_j$ (division by $N$, not $N-1$ — the same convention adopted by the `StandardScaler` class) of each feature $j \in \{1,2\}$, **exclusively on the training set**.
6. **Standardization:** Transform each training and query feature by
$$
z_j = \frac{x_j - \mu_j}{\sigma_j}.
$$
If $\sigma_j = 0$ (constant feature in the training set), set $z_j = 0$ for all samples of that feature, avoiding division by zero.
7. **Classification with normalization:** Repeat the k-NN classification of item 4, now on the standardized vectors $(z_1, z_2)$, with the same tie-breaking rules.
8. **Output:** For each query, in input order, print the two predicted classes. At the end, print the number of queries in which the two classifications **diverge**.

#### 📌 Computational Constraints

* **Fit on training data only:** $\mu_j$ and $\sigma_j$ are computed solely from the training set and reapplied to the queries — never recalculated from them. This practice avoids **data leakage**, mentioned in the normalization section of the chapter.
* **Population standard deviation:** use $\sigma_j = \sqrt{\frac{1}{N}\sum_i (x_{i,j}-\mu_j)^2}$, not the sample version (division by $N-1$).
* **Constant feature:** treat $\sigma_j = 0$ as a special case (item 6); no division-by-zero error should occur.
* **Tie-breaking rules:** reuse exactly the conventions from EP07_01, both in selecting the $k$ neighbors and in the majority vote.

#### 🧠 Theoretical Foundation

| Element | Role |
|---|---|
| *Z-score* standardization | Rescales each feature to mean $0$ and standard deviation $1$, making heterogeneous scales comparable |
| Fit on training data only | Ensures that evaluation on queries reflects only what the model learned from the training data |
| Euclidean distance without normalization | Dominated by the feature with the largest magnitude — here, the area |
| Divergent prediction | Highlights that the scale of features, not just the algorithm or the data, can determine the k-NN decision boundary |

This exercise reinforces, in a controlled manner, the reason why `StandardScaler` is applied before k-NN throughout the chapter: without this step, circularity features — even when highly discriminative — can be practically ignored by the classifier in the presence of an area feature with a magnitude hundreds of times larger.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $N$ and $k$, separated by a space.
* Next $N$ lines: three values per line — $x_1$, $x_2$ (real) and $r$ (integer $\in \{0,1\}$), separated by a space.
* Next line: integer $Q$.
* Next $Q$ lines: two values per line — $x_1$, $x_2$ (real) of the query, separated by a space.

**Output:**

* $Q$ lines, in the format `SemNorm=<0|1> ComNorm=<0|1>`, in the input order of the queries.
* Last line: `Divergiu: <int>`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4 3<br>10 0.9 0<br>12 0.85 0<br>900 0.2 1<br>950 0.25 1<br>1<br>500 0.88 | SemNorm=1 ComNorm=0<br>Divergiu: 1 | Without normalization, the area (scale of hundreds) dominates the distance, and the query is classified as class `1`. After standardization, the circularity — much closer to the class `0` samples — becomes comparably weighted, and the prediction changes to `0`. |
| 2 1<br>0 0.5 0<br>100 0.5 1<br>1<br>60 0.5 | SemNorm=1 ComNorm=1<br>Divergiu: 0 | The circularity is constant in the training set ($\sigma_2=0$); by the rule of item 6, $z_2=0$ for all samples, and the classification depends only on the area in both cases. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0702" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0702 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0702 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0702 button:hover { background: #e8dfcf; }
  #sim-ep0702 button.sim-ep0702_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0702_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0702_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP07_02: Z-score Normalization and k-NN Distance</span>
  <span class="sim-ep0702_pill">Feature Standardization</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Seleção de Modo -->
  <div class="sim-ep0702_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:8px;">
      Each example has two features: area (px) and circularity [0, 1]. Toggle normalization and observe the change in the predicted class.
    </div>
    
    <div id="sim-ep0702_query" style="font-size:11px; color:#26241d; text-align:center; font-family:monospace; font-weight:700; margin-bottom:10px;"></div>

    <div style="display:flex; justify-content:center; gap:8px; flex-wrap:wrap;">
      <button id="sim-ep0702_btn_raw" class="sim-ep0702_active">Without Normalization</button>
      <button id="sim-ep0702_btn_norm">With Normalization (Z-score)</button>
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0702_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0702_debug" class="sim05_ep01_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep02(root){
    if (!root || root.dataset.sim07Ep02Init) return;
    root.dataset.sim07Ep02Init = "1";

    var pontos = [
      {nome: "P1", x1: 10,  x2: 0.90, r: 0},
      {nome: "P2", x1: 12,  x2: 0.85, r: 0},
      {nome: "P3", x1: 900, x2: 0.20, r: 1},
      {nome: "P4", x1: 950, x2: 0.25, r: 1}
    ];

    pontos.forEach(function(p, i){ p.idx = i; });
    var query = {x1: 500, x2: 0.88};
    var k = 3;

    function stats(vals){
      var m = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
      var v = vals.reduce(function(a, s){ return a + (s - m) * (s - m); }, 0) / vals.length;
      return {mean: m, std: Math.sqrt(v)};
    }

    var s1 = stats(pontos.map(function(p){ return p.x1; }));
    var s2 = stats(pontos.map(function(p){ return p.x2; }));

    function z(x, s){ return s.std === 0 ? 0 : (x - s.mean) / s.std; }

    var cards   = root.querySelector('#sim-ep0702_cards');
    var dbg     = root.querySelector('#sim-ep0702_debug');
    var qEl     = root.querySelector('#sim-ep0702_query');
    var btnRaw  = root.querySelector('#sim-ep0702_btn_raw');
    var btnNorm = root.querySelector('#sim-ep0702_btn_norm');
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle('sim-ep0702_active', !modoNorm);
      btnNorm.classList.toggle('sim-ep0702_active', modoNorm);

      qEl.textContent = '★ Query: Area = ' + query.x1 + ', Circularidade = ' + query.x2 +
        (modoNorm ? ' → z_área = ' + z(query.x1, s1).toFixed(3) + ', z_circ = ' + z(query.x2, s2).toFixed(3) : '');

      var qx1 = modoNorm ? z(query.x1, s1) : query.x1;
      var qx2 = modoNorm ? z(query.x2, s2) : query.x2;

      var lista = pontos.map(function(p){
        var px1 = modoNorm ? z(p.x1, s1) : p.x1;
        var px2 = modoNorm ? z(p.x2, s2) : p.x2;
        var d = Math.sqrt((px1 - qx1) * (px1 - qx1) + (px2 - qx2) * (px2 - qx2));
        return {nome: p.nome, r: p.r, d: d, idx: p.idx, area: p.x1, circ: p.x2, va: px1, vc: px2};
      });

      lista.sort(function(a, b){ return (a.d - b.d) || (a.idx - b.idx); });

      cards.innerHTML = '';
      var votos = [0, 0];

      lista.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        var valorUsado = modoNorm
          ? ('z = (' + p.va.toFixed(2) + ', ' + p.vc.toFixed(2) + ')')
          : ('área = ' + p.area + ', circ = ' + p.circ);

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">' + valorUsado + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px;">d = ' + p.d.toFixed(3) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : lista[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = (modoNorm ? 'COM Normalização' : 'SEM Normalização') +
        '  |  k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] +
        '  |  Classe prevista: ' + previsto;
    }

    btnRaw.addEventListener('click', function(){ modoNorm = false; render(); });
    btnNorm.addEventListener('click', function(){ modoNorm = true; render(); });
    render();
  }

  function tryInitSim07Ep02(){
    var root = document.getElementById('sim-ep0702');
    if (root) initSim07Ep02(root); else setTimeout(tryInitSim07Ep02, 200);
  }
  tryInitSim07Ep02();
})();
</script>
""")

**Figura 7.2:** EP07_02 Simulator: Effect of *Z-score* Normalization on k-NN Distance


<figure id="fig-07-sim-ep0702">
  <img src="imagens/fig-07-sim-ep0702.png" alt=" EP07_02 Simulator: Effect of *Z-score* Normalization on k-NN Distance " style="max-width:80%" />
  <figcaption><strong>Figura 7.2:</strong>  EP07_02 Simulator: Effect of *Z-score* Normalization on k-NN Distance </figcaption>
</figure>

In [ ]:
%%writefile EP07_02.py
# Python code

In [ ]:
TestSuite("EP07_02.py").run()

### EP07_03 🟡 Evaluation via Confusion Matrix

A binary weld quality classifier was trained and tested on a production line. For each inspected part, the system recorded the **actual** label (obtained by an expert) and the **predicted** label from the classifier, where `1` represents "defective" and `0` represents "conforming."

Quality management wants to know not only the system's accuracy but also its **precision** (when the system flags a defect, how often is it correct?) and its **recall** (of all truly defective parts, how many did the system manage to identify?)—the distinction discussed in the classifier evaluation section of the chapter.

#### 📋 Implementation Guidelines

1. **Quantity:** Read the integer $N$ (number of inspected parts).
2. **Data for each part:** For each of the $N$ parts, read two integers—the actual label $y$ and the predicted label $\hat y$ (both $\in \{0, 1\}$).
3. **Confusion matrix:** Considering class `1` (defective) as **positive**, count:
   - $VP$ (True Positive): $y=1$ and $\hat y=1$;
   - $FP$ (False Positive): $y=0$ and $\hat y=1$;
   - $FN$ (False Negative): $y=1$ and $\hat y=0$;
   - $VN$ (True Negative): $y=0$ and $\hat y=0$.
4. **Metrics:** Calculate
$$
\text{Accuracy} = \frac{VP+VN}{N}, \quad
\text{Precision} = \frac{VP}{VP+FP}, \quad
\text{Recall} = \frac{VP}{VP+FN}.
$$
5. **Degenerate cases:** If $VP+FP=0$ (no positive predictions), print `Precisao: indefinida`. If $VP+FN=0$ (no actual positive cases), print `Revocacao: indefinida`.
6. **Rounding:** All numerical metrics must be rounded to 4 decimal places (*round half away from zero*) only for display.

#### 📌 Computational Constraints

* **Fixed positive class convention:** class `1` is always the positive class in this exercise, regardless of its relative frequency.
* **Division by zero protection:** implement the degenerate cases from item 5 before performing the division.
* **Output order:** follow exactly the order specified in the output section, even in degenerate cases.

#### 🧠 Theoretical Foundation

| Metric | Question It Answers | Sensitive to Imbalance? |
|---|---|---|
| Accuracy | What fraction of parts was classified correctly? | Yes—can mask errors in the minority class |
| Precision | Of the parts flagged as defective, how many truly are? | Penalizes false positives |
| Recall | Of the truly defective parts, how many were detected? | Penalizes false negatives |

In an industrial context, **low recall** is often more severe than **low precision**: letting a defective part pass (false negative) tends to be costlier than manually inspecting a good part flagged by mistake (false positive).

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $N$.
* Next $N$ lines: two integers per line—$y$ and $\hat y$, separated by spaces.

**Output (in this exact order):**

```
VP=<int> FP=<int> FN=<int> VN=<int>
Acuracia: <value or undefined metric>
Precisao: <value or undefined>
Revocacao: <value or undefined>
```

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4<br>1 1<br>0 1<br>1 0<br>0 0 | VP=1 FP=1 FN=1 VN=1<br>Acuracia: 0.5000<br>Precisao: 0.5000<br>Revocacao: 0.5000 | One error of each type. |
| 3<br>0 0<br>0 0<br>0 0 | VP=0 FP=0 FN=0 VN=3<br>Acuracia: 1.0000<br>Precisao: indefinida<br>Revocacao: indefinida | No actual or predicted positive cases. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0703" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0703 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0703 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0703 button:hover { background: #e8dfcf; }
  #sim-ep0703 button.sim-ep0703_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0703_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0703_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 EP07_03 Simulator: Precision x Recall</span>
  <span class="sim-ep0703_pill">Production Line</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Cenário -->
  <div class="sim-ep0703_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Choose an inspection scenario and observe how Accuracy, Precision, and Recall react differently.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0703_b1" class="sim-ep0703_active">Scenario A: Balanced Errors</button>
      <button id="sim-ep0703_b2">Scenario B: False Negatives</button>
      <button id="sim-ep0703_b3">Scenario C: No Real Defect</button>
      <button id="sim-ep0703_b4">Scenario D: False Positives</button>
    </div>
  </div>

  <!-- Cards das Peças do Cenário -->
  <div id="sim-ep0703_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0703_debug" class="sim-ep0703_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep03(root){
    if (!root || root.dataset.sim07Ep03Init) return;
    root.dataset.sim07Ep03Init = "1";

    var cenarios = {
      A: [{y:1, p:1}, {y:0, p:1}, {y:1, p:0}, {y:0, p:0}],
      B: [{y:1, p:0}, {y:1, p:0}, {y:1, p:1}, {y:0, p:0}],
      C: [{y:0, p:0}, {y:0, p:0}, {y:0, p:0}],
      D: [{y:0, p:1}, {y:0, p:1}, {y:1, p:1}, {y:0, p:0}]
    };

    var cards = root.querySelector('#sim-ep0703_cards');
    var dbg   = root.querySelector('#sim-ep0703_debug');

    var botoes = {
      A: root.querySelector('#sim-ep0703_b1'),
      B: root.querySelector('#sim-ep0703_b2'),
      C: root.querySelector('#sim-ep0703_b3'),
      D: root.querySelector('#sim-ep0703_b4')
    };

    function render(key){
      Object.keys(botoes).forEach(function(k){
        botoes[k].classList.toggle('sim-ep0703_active', k === key);
      });

      var dados = cenarios[key];
      var VP = 0, FP = 0, FN = 0, VN = 0;
      cards.innerHTML = '';

      dados.forEach(function(d, i){
        if (d.y === 1 && d.p === 1) VP++;
        else if (d.y === 0 && d.p === 1) FP++;
        else if (d.y === 1 && d.p === 0) FN++;
        else VN++;

        var statusCor = '';
        var statusTxt = '';

        if (d.y === d.p) {
          statusCor = 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
          statusTxt = d.y === 1 ? 'VP (Acerto)' : 'VN (Acerto)';
        } else {
          statusCor = 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;';
          statusTxt = d.p === 1 ? 'FP (Alarme Falso)' : 'FN (Escapou)';
        }

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' + statusCor;

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">Peça ' + (i + 1) + '</div>' +
          '<div style="font-family:monospace; font-size:10px; margin-bottom:4px;">Real = ' + d.y + ' | Prev = ' + d.p + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + statusTxt + '</div>';

        cards.appendChild(div);
      });

      var N = dados.length;
      var acc = ((VP + VN) / N).toFixed(4);
      var prec = (VP + FP) > 0 ? (VP / (VP + FP)).toFixed(4) : 'Indefinida';
      var rev = (VP + FN) > 0 ? (VP / (VP + FN)).toFixed(4) : 'Indefinida';

      if (prec === 'Indefinida' || parseFloat(prec) < 0.5) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'TP = ' + VP + ' | FP = ' + FP + ' | FN = ' + FN + ' | VN = ' + VN +
        '  |  Acurácia = ' + acc + '  |  Precisão = ' + prec + '  |  Revocação = ' + rev;
    }

    botoes.A.addEventListener('click', function(){ render('A'); });
    botoes.B.addEventListener('click', function(){ render('B'); });
    botoes.C.addEventListener('click', function(){ render('C'); });
    botoes.D.addEventListener('click', function(){ render('D'); });

    render('A');
  }

  function tryInitSim07Ep03(){
    var root = document.getElementById('sim-ep0703');
    if (root) initSim07Ep03(root); else setTimeout(tryInitSim07Ep03, 200);
  }
  tryInitSim07Ep03();
})();
</script>
""")

**Figura 7.3:** Simulator EP07_03: Precision x Recall


<figure id="fig-07-sim-ep0703">
  <img src="imagens/fig-07-sim-ep0703.png" alt=" Simulator EP07_03: Precision x Recall " style="max-width:80%" />
  <figcaption><strong>Figura 7.3:</strong>  Simulator EP07_03: Precision x Recall </figcaption>
</figure>

In [ ]:
%%writefile EP07_03.py
# Python code

In [ ]:
TestSuite("EP07_03.py").run()

### EP07_04 🟠 Manual Encoding of the LBP Descriptor

The `local_binary_pattern` function from `scikit-image`, used in the texture classification project, automatically computes the LBP code for each pixel of an image. Before using it as a black box, you have been tasked with manually implementing the computation of the classic LBP code ($P=8$, $R=1$) for the central pixel of a $3\times3$ neighborhood, exactly as defined in the equation in the chapter.

In addition to the code, the texture inspection system also needs to know whether that pattern is **uniform** — a pattern is uniform when the number of transitions ($0\to1$ or $1\to0$) when traversing the 8 bits **circularly** (returning from the last bit to the first) is **at most 2**, a property exploited by the *uniform* variant of the LBP mentioned in the chapter.

#### 📋 Implementation Guidelines

1. **Quantity:** Read the integer $T$ (number of neighborhoods to process).
2. **Data for each neighborhood:** For each of the $T$ neighborhoods, read a $3\times3$ matrix of integers (intensities), provided in 3 lines of 3 values each. The central pixel is at position `[1][1]`.
3. **Order of neighbors:** Traverse the 8 neighbors in **clockwise** order, starting at the top-left corner, in the following sequence of positions `[row][column]`: `[0][0]`, `[0][1]`, `[0][2]`, `[1][2]`, `[2][2]`, `[2][1]`, `[2][0]`, `[1][0]`. This corresponds to the index $p = 0, 1, \ldots, 7$ in the LBP equation.
4. **Threshold function:** For each neighbor $p$ with intensity $g_p$ and center $g_c$, compute $s(g_p - g_c)$, which is `1` if $g_p \geq g_c$ and `0` otherwise.
5. **LBP code:** Compute
$$
\mathrm{LBP} = \sum_{p=0}^{7} s(g_p - g_c)\, 2^p.
$$
6. **Transitions:** Considering the circular bit sequence $s_0, s_1, \ldots, s_7$ (in the order from item 3), count how many consecutive pairs **adjacent in the circular sequence** (including the pair $s_7, s_0$) differ from each other.
7. **Classification:** If the number of transitions is $\le 2$, classify as `UNIFORME`; otherwise, classify as `NAO_UNIFORME`.
8. **Output:** For each neighborhood, in input order, print the LBP code (decimal integer, $0$–$255$), the number of transitions, and the classification.

#### 📌 Computational Constraints

* **Fixed neighbor order:** The order from item 3 is mandatory — reversing it produces a numerically different code, even though it represents the same visual pattern.
* **Non-strict comparison:** $s(z) = 1$ when $z \ge 0$ (the chapter itself defines equality as included in the `1` case).
* **Circular counting:** Do not forget the pair that closes the cycle ($s_7$ with $s_0$); ignoring this pair is a common mistake that incorrectly classifies uniform patterns.

#### 🧠 Theoretical Foundation

| Pattern (bits $s_0\ldots s_7$) | Transitions | Interpretation |
|---|---|---|
| `00000000` or `11111111` | 0 | Homogeneous region (light or dark patch) |
| `00001111` | 2 | Simple edge between two regions |
| `01010101` | 8 | Alternating contrast texture — non-uniform |

Uniform patterns are concentrated in smooth texture regions or simple edges; non-uniform patterns tend to correspond to high-frequency noise. For this reason, the *uniform* LBP histogram, used in the texture classification project, groups all non-uniform patterns into a single bin, reducing the dimensionality of the descriptor.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $T$.
* For each neighborhood: 3 lines with 3 integers each ($3\times3$ matrix).

**Output:**

* $T$ lines, in the format `LBP=<int> transicoes=<int> <UNIFORME|NAO_UNIFORME>`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 1<br>10 10 10<br>10 50 10<br>10 10 10 | LBP=0 transicoes=0 UNIFORME | Center is the brightest; all neighbors generate bit 0. |
| 1<br>90 90 90<br>10 50 10<br>90 90 90 | LBP=119 transicoes=4 NAO_UNIFORME | Light and dark neighbors alternate in the neighborhood. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0704" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0704 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0704 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0704 button:hover { background: #e8dfcf; }
  #sim-ep0704 button.sim-ep0704_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0704_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0704_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP07_04: LBP Code of a Neighborhood 3&times;3</span>
  <span class="sim-ep0704_pill">P = 8, R = 1</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Exemplo -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Click a cell in the neighborhood to toggle between light and dark (the center is fixed) and observe the resulting LBP code. The label p indicates the equation index.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0704_b1">Example 1: Homogeneous Patch</button>
      <button id="sim-ep0704_b2" class="sim-ep0704_active">Example 2: Alternating Pattern</button>
    </div>
  </div>

  <!-- Grid Vizinhança 3x3 -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px; text-align:center;">
    <div id="sim-ep0704_grid" style="display:grid; grid-template-columns:repeat(3, 60px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0704_debug" class="sim-ep0704_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep04(root){
    if (!root || root.dataset.sim07Ep04Init) return;
    root.dataset.sim07Ep04Init = "1";

    var exemplos = {
      1: [[10, 10, 10], [10, 50, 10], [10, 10, 10]],
      2: [[90, 90, 90], [10, 50, 10], [90, 90, 90]]
    };

    var valores = exemplos[2].map(function(row){ return row.slice(); });
    var grid = root.querySelector('#sim-ep0704_grid');
    var dbg  = root.querySelector('#sim-ep0704_debug');
    var btn1 = root.querySelector('#sim-ep0704_b1');
    var btn2 = root.querySelector('#sim-ep0704_b2');

    var ordem = [[0, 0], [0, 1], [0, 2], [1, 2], [2, 2], [2, 1], [2, 0], [1, 0]];
    var pIndex = {};
    ordem.forEach(function(pos, p){ pIndex[pos[0] + ',' + pos[1]] = p; });
    var cenarioAtivo = 2;

    function marcarBotaoAtivo(n){
      cenarioAtivo = n;
      btn1.classList.toggle('sim-ep0704_active', n === 1);
      btn2.classList.toggle('sim-ep0704_active', n === 2);
    }

    function render(){
      grid.innerHTML = '';
      for (var r = 0; r < 3; r++){
        for (var c = 0; c < 3; c++){
          (function(r, c){
            var v = valores[r][c];
            var central = (r === 1 && c === 1);
            var div = document.createElement('div');

            var bordaCor = central ? '#26241d' : '#e4dcc8';
            var textoCor = v > 128 ? '#26241d' : '#ffffff';

            div.style.cssText = 'position:relative; height:60px; display:flex; align-items:center; justify-content:center; font-family:monospace; font-weight:700; border-radius:6px; cursor:' + (central ? 'default' : 'pointer') + '; border:2px solid ' + bordaCor + '; background:rgb(' + v + ',' + v + ',' + v + '); color:' + textoCor + '; transition:all 0.15s ease;';
            div.textContent = v;

            if (!central){
              var pLabel = document.createElement('span');
              pLabel.textContent = 'p' + pIndex[r + ',' + c];
              pLabel.style.cssText = 'position:absolute; top:2px; left:4px; font-size:9px; font-weight:400; opacity:0.8;';
              div.appendChild(pLabel);

              div.addEventListener('click', function(){
                valores[r][c] = valores[r][c] >= 128 ? 10 : 200;
                cenarioAtivo = null;
                btn1.classList.remove('sim-ep0704_active');
                btn2.classList.remove('sim-ep0704_active');
                render();
              });
            }
            grid.appendChild(div);
          })(r, c);
        }
      }

      var gc = valores[1][1];
      var bits = ordem.map(function(pos){ return valores[pos[0]][pos[1]] >= gc ? 1 : 0; });
      var lbp = 0;
      bits.forEach(function(b, p){ lbp += b * Math.pow(2, p); });

      var trans = 0;
      for (var i = 0; i < 8; i++){
        if (bits[i] !== bits[(i + 1) % 8]) trans++;
      }

      var classe = trans <= 2 ? 'UNIFORME' : 'NÃO-UNIFORME';

      if (trans <= 2) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'bits (p0..p7) = ' + bits.join('') + '  |  LBP = ' + lbp + '  |  transições = ' + trans + '  |  ' + classe;
    }

    btn1.addEventListener('click', function(){
      valores = exemplos[1].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(1);
      render();
    });

    btn2.addEventListener('click', function(){
      valores = exemplos[2].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(2);
      render();
    });

    marcarBotaoAtivo(2);
    render();
  }

  function tryInitSim07Ep04(){
    var root = document.getElementById('sim-ep0704');
    if (root) initSim07Ep04(root); else setTimeout(tryInitSim07Ep04, 200);
  }
  tryInitSim07Ep04();
})();
</script>
""")

**Figura 7.4:** EP07_04 Simulator: LBP Code of a 3x3 Neighborhood


<figure id="fig-07-sim-ep0704">
  <img src="imagens/fig-07-sim-ep0704.png" alt=" EP07_04 Simulator: LBP Code of a 3x3 Neighborhood " style="max-width:80%" />
  <figcaption><strong>Figura 7.4:</strong>  EP07_04 Simulator: LBP Code of a 3x3 Neighborhood </figcaption>
</figure>

In [ ]:
%%writefile EP07_04.py
# Python code

In [ ]:
TestSuite("EP07_04.py").run()

### EP07_05 🔴 Histogram of Orientations for an HOG Cell

The `hog` function from `scikit-image`, used in the digit classification project, divides the image into small **cells** and, for each one, builds a histogram of gradient orientations weighted by magnitude — exactly the central step described in the section on the HOG descriptor in the chapter.

You have been tasked with implementing this computation for a single cell, using the magnitude and gradient orientation values **already computed** for each pixel in the cell (dispensing with the computation of partial derivatives).

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $n$ (the cell has $n \times n$ pixels) and $B$ (number of histogram bins).
2. **Magnitudes:** Read $n$ lines with $n$ real values each, representing $|\nabla f(x,y)|$ for each pixel in the cell.
3. **Orientations:** Read $n$ more lines with $n$ real values each, representing $\theta(x,y)$ in **degrees**, already converted to the **unsigned** interval $[0^\circ, 180^\circ)$, as conventionally used by HOG.
4. **Bins:** The $B$ bins cover $[0^\circ, 180^\circ)$ in equal bands of width $180/B$ degrees. A pixel with orientation $\theta$ belongs to bin $\lfloor \theta / (180/B) \rfloor$; if this index equals $B$ (possible only when $\theta$ is exactly $180^\circ$, which should not occur according to the guideline in item 3), use bin $B-1$.
5. **Raw histogram:** For each pixel, accumulate its **magnitude** (not its count) in the corresponding bin:
$$
H[b] = \sum_{(x,y)\, :\, \text{bin}(\theta(x,y)) = b} |\nabla f(x,y)|.
$$
6. **L2 normalization:** After constructing $H$, normalize it to obtain $\hat H$:
$$
\hat H[b] = \frac{H[b]}{\sqrt{\sum_{j=0}^{B-1} H[j]^2 + \epsilon}}, \qquad \epsilon = 10^{-6}.
$$
7. **Output:** Print the raw histogram $H$ (rounded to 2 decimal places) on one line, followed by the normalized histogram $\hat H$ (rounded to 4 decimal places) on another line, both with the $B$ values separated by spaces, in bin order.

#### 📌 Computational Constraints

* ***Unsigned binning*:** the orientation interval is $[0,180)$, not $[0,360)$ — gradients in opposite directions (differing by $180^\circ$) contribute to the **same** bin, the standard HOG convention for object detection.
* **Accumulation by magnitude, not by count:** the histogram weights each pixel by its gradient magnitude, rather than simply counting how many pixels fall into each bin.
* **Stabilization constant:** the $\epsilon = 10^{-6}$ in the normalization denominator avoids division by zero when the cell is completely homogeneous (all magnitudes zero).

#### 📐 Where the Input Matrices Come From

Before this assignment, each pixel $(x,y)$ of the image undergoes:

$$
G_x = f(x+1,y)-f(x-1,y), \qquad G_y = f(x,y+1)-f(x,y-1)
$$

$$
|\nabla f| = \sqrt{G_x^2+G_y^2}, \qquad \theta_{\text{signed}} = \operatorname{atan2}(G_y,G_x)
$$

Since HOG disregards contrast polarity, the angle is folded into the unsigned interval:

$$
\theta = \theta_{\text{signed}} \bmod 180°
$$

Repeating this for all pixels in an $n\times n$ cell yields the two input matrices for this exercise: **magnitudes** $|\nabla f|$ and **orientations** $\theta \in [0°,180°)$.

#### 🧠 Theoretical Background

| Step | Role |
|---|---|
| Gradient magnitude | Weights each pixel's contribution — strong edges weigh more than weak noise |
| Unsigned orientation | Makes the descriptor invariant to contrast polarity (light→dark vs. dark→light) |
| Per-cell histogram | Summarizes the local edge distribution into a compact vector |
| L2 normalization | Reduces the descriptor's sensitivity to global illumination and contrast variations |

The concatenation of the normalized histograms from all cells in the image — not implemented in this exercise — forms the complete HOG feature vector, used as input to the k-NN classifier in the chapter's project.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $n$ and $B$.
* Next $n$ lines: $n$ real magnitudes each.
* Next $n$ lines: $n$ real orientations (degrees, $[0,180)$) each.

**Output:**

* Line 1: the $B$ values of the raw histogram, rounded to 2 decimal places.
* Line 2: the $B$ values of the normalized histogram, rounded to 4 decimal places.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 2 2<br>1.0 2.0<br>3.0 4.0<br>10 100<br>170 20 | 5.00 5.00<br>0.7071 0.7071 | Bin width 90°: $[0,90)$ and $[90,180)$; magnitudes 1 and 4 fall into bin 0, 2 and 3 into bin 1. |
| 2 4<br>0.0 0.0<br>0.0 0.0<br>0 0<br>0 0 | 0.00 0.00 0.00 0.00<br>0.0000 0.0000 0.0000 0.0000 | Homogeneous cell: $\epsilon$ avoids division by zero. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0705" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0705 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0705 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0705 button:hover { background: #e8dfcf; }
  #sim-ep0705 button.sim-ep0705_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0705_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0705_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP07_05: Orientation Histogram of a Cell</span>
  <span class="sim-ep0705_pill">🔴 fixed 3×3 cell</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Adjust B and watch how the <b>orientation matrix</b> (independent of the magnitudes one) is mapped
      to the bins via <code>bin = floor(θ / (180/B))</code>, and how the magnitudes are summed into each bin.
    </p>

    <!-- Controle B -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Number of bins (B)</label>
        <span id="ep0705_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">2</span>
      </div>
      <input id="ep0705_sl" style="width:100%;accent-color:#2980b9;" max="6" min="2" step="1" type="range" value="2">
    </div>

    <!-- Entrada bruta (formato VPL) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📄 Input (exactly as the program reads from stdin)</div>
      <pre id="ep0705_stdin" style="background:#1e1e1e;color:#d4d4d4;border-radius:8px;padding:12px 14px;font-size:12px;line-height:1.5;overflow-x:auto;margin:0;"></pre>
    </div>

    <!-- Duas matrizes separadas -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:20px;">
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🔢 Magnitude matrix |∇f|</div>
        <div id="ep0705_mag_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📐 Orientation matrix θ (degrees) — colored by bin</div>
        <div id="ep0705_ang_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
    </div>

    <!-- Regua 0-180 -->
    <div style="margin-bottom:22px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:10px;">📏 Where each θ falls on the ruler [0°, 180°) — <code>bin = floor(θ / width)</code></div>
      <div style="position:relative;height:70px;margin:0 6px;">
        <div id="ep0705_regua" style="position:absolute;top:28px;left:0;right:0;height:14px;border-radius:7px;overflow:hidden;display:flex;border:1px solid #d1d5db;"></div>
        <div id="ep0705_regua_ticks" style="position:absolute;top:44px;left:0;right:0;height:14px;"></div>
        <div id="ep0705_regua_marcas" style="position:absolute;top:0;left:0;right:0;height:26px;"></div>
      </div>
    </div>

    <!-- Faixas dos compartimentos -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📊 Range of each bin (width = 180° / B)</div>
      <div id="ep0705_faixas" style="display:flex;flex-wrap:wrap;gap:6px;"></div>
    </div>

    <!-- Grade de pixels colorida por bin (mag + ang juntos) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧩 Each pixel: magnitude + orientation → bin</div>
      <div id="ep0705_pixels" style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <!-- Botões -->
    <div style="display:flex;gap:8px;justify-content:center;margin-bottom:14px;">
      <button id="ep0705_btn_raw" class="ep0705_btn">Raw histogram (H)</button>
      <button id="ep0705_btn_norm" class="ep0705_btn">Normalized histogram (Ĥ)</button>
    </div>

    <!-- Barras -->
    <div id="ep0705_bars" style="display:flex;gap:6px;align-items:flex-end;height:120px;justify-content:center;margin-bottom:14px;"></div>

    <!-- Passo a passo -->
    <div style="margin-bottom:6px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧮 Step-by-step calculation (floor of the division + summing magnitudes per bin)</div>
      <div id="ep0705_passos" style="background:#f3f4f6;border-radius:8px;padding:10px 12px;font-family:monospace;font-size:11px;color:#374151;line-height:1.8;"></div>
    </div>

    <div id="ep0705_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0705 .ep0705_btn { font-size:11px;padding:6px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0705 .ep0705_btn.ativo { background:#2980b9;color:#fff;border-color:#2980b9; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var n = 3;
    var mags = [[1.0,2.0,0.5],[3.0,4.0,1.5],[0.8,2.5,3.2]];
    var angs = [[10,100,45],[170,20,95],[60,150,5]];
    var CORES = ["#6366f1","#0ea5e9","#10b981","#f59e0b","#ef4444","#a855f7"];

    var slEl = root.querySelector("#ep0705_sl");
    var vlEl = root.querySelector("#ep0705_vl");
    var stdinEl = root.querySelector("#ep0705_stdin");
    var magGridEl = root.querySelector("#ep0705_mag_grid");
    var angGridEl = root.querySelector("#ep0705_ang_grid");
    var reguaEl = root.querySelector("#ep0705_regua");
    var reguaTicksEl = root.querySelector("#ep0705_regua_ticks");
    var reguaMarcasEl = root.querySelector("#ep0705_regua_marcas");
    var faixasEl = root.querySelector("#ep0705_faixas");
    var pxEl = root.querySelector("#ep0705_pixels");
    var bars = root.querySelector("#ep0705_bars");
    var passosEl = root.querySelector("#ep0705_passos");
    var dbg = root.querySelector("#ep0705_debug");
    var btnRaw = root.querySelector("#ep0705_btn_raw");
    var btnNorm = root.querySelector("#ep0705_btn_norm");
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle("ativo", !modoNorm);
      btnNorm.classList.toggle("ativo", modoNorm);

      var B = parseInt(slEl.value);
      vlEl.textContent = B;
      var largura = 180/B;

      // ---- Entrada bruta (stdin) ----
      var linhas = [];
      linhas.push(n + " " + B);
      mags.forEach(function(row){ linhas.push(row.map(function(v){return v.toFixed(1);}).join(" ")); });
      angs.forEach(function(row){ linhas.push(row.join(" ")); });
      stdinEl.textContent = linhas.join("\\n");

      // ---- bin de cada pixel (floor(theta/largura), clip) ----
      var binsMat = [];
      for(var i=0;i<n;i++){
        binsMat.push([]);
        for(var j=0;j<n;j++){
          var raw = angs[i][j]/largura;
          var b = Math.floor(raw);
          if(b > B-1) b = B-1;
          if(b < 0) b = 0;
          binsMat[i].push(b);
        }
      }

      // ---- Matriz de magnitudes (grid simples) ----
      magGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var d = document.createElement("div");
          d.style.cssText = "text-align:center;border-radius:8px;padding:8px 4px;font-size:12px;font-family:monospace;background:#f9fafb;border:1px solid #e5e7eb;color:#374151;";
          d.textContent = mags[i][j].toFixed(1);
          magGridEl.appendChild(d);
        }
      }

      // ---- Matriz de orientações (colorida por bin, com floor explícito) ----
      angGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var b2 = binsMat[i][j];
          var cor2 = CORES[b2];
          var raw2 = angs[i][j]/largura;
          var d2 = document.createElement("div");
          d2.style.cssText = "text-align:center;border-radius:8px;padding:6px 4px;font-size:11px;font-family:monospace;background:"+cor2+"22;border:2px solid "+cor2+";color:#374151;";
          d2.innerHTML = "<div style=\\"font-weight:700;\\">"+angs[i][j]+"°</div>"+
            "<div style=\\"font-size:9px;color:#6b7280;\\">÷"+largura.toFixed(1)+"="+raw2.toFixed(2)+"</div>"+
            "<div style=\\"font-size:9px;font-weight:700;color:"+cor2+";\\">⌊·⌋=bin "+b2+"</div>";
          angGridEl.appendChild(d2);
        }
      }

      // ---- Régua 0-180 com faixas coloridas ----
      reguaEl.innerHTML = "";
      for(var b3=0;b3<B;b3++){
        var seg = document.createElement("div");
        seg.style.cssText = "flex:1;background:"+CORES[b3]+";opacity:0.35;border-right:1px solid rgba(255,255,255,0.6);";
        reguaEl.appendChild(seg);
      }
      // ticks (limites dos bins)
      reguaTicksEl.innerHTML = "";
      for(var b4=0;b4<=B;b4++){
        var pct = (b4*largura/180*100);
        var tick = document.createElement("div");
        tick.style.cssText = "position:absolute;left:"+pct+"%;top:0;font-size:9px;color:#6b7280;transform:translateX(-50%);white-space:nowrap;";
        tick.textContent = (b4*largura).toFixed(0)+"°";
        reguaTicksEl.appendChild(tick);
      }
      // marcadores dos angulos de cada pixel
      reguaMarcasEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var ang = angs[i][j];
          var b5 = binsMat[i][j];
          var pctm = (ang/180*100);
          var marker = document.createElement("div");
          marker.style.cssText = "position:absolute;left:"+pctm+"%;top:0;transform:translateX(-50%);display:flex;flex-direction:column;align-items:center;";
          marker.innerHTML = "<div style=\\"font-size:9px;color:"+CORES[b5]+";font-weight:700;\\">("+i+","+j+")</div>"+
            "<div style=\\"width:0;height:0;border-left:5px solid transparent;border-right:5px solid transparent;border-top:8px solid "+CORES[b5]+";\\"></div>";
          reguaMarcasEl.appendChild(marker);
        }
      }

      // ---- Faixas dos bins (legenda) ----
      faixasEl.innerHTML = "";
      for(var b=0;b<B;b++){
        var lo = (b*largura).toFixed(1);
        var hi = ((b+1)*largura).toFixed(1);
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:6px;background:#f9fafb;border:1px solid #e5e7eb;border-radius:20px;padding:4px 10px;font-size:11px;color:#374151;";
        chip.innerHTML = "<span style=\\"width:10px;height:10px;border-radius:50%;background:"+CORES[b]+";display:inline-block;\\"></span>bin "+b+": ["+lo+"°, "+hi+"°)";
        faixasEl.appendChild(chip);
      }

      // ---- Atribuição por pixel + histograma bruto ----
      var H = new Array(B).fill(0);
      var binsPorPixel = [];
      pxEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var mag = mags[i][j], ang = angs[i][j];
          var bin = binsMat[i][j];
          binsPorPixel.push({i:i, j:j, mag:mag, ang:ang, bin:bin});
          H[bin] += mag;

          var div = document.createElement("div");
          var cor = CORES[bin];
          div.style.cssText = "text-align:center;border-radius:10px;padding:8px 6px;font-size:11px;background:"+cor+"22;border:2px solid "+cor+";color:#374151;";
          div.innerHTML = "<div style=\\"font-weight:700;\\">mag="+mag.toFixed(1)+"</div>"+
            "<div style=\\"font-family:monospace;\\">θ="+ang+"°</div>"+
            "<div style=\\"font-weight:700;color:"+cor+";\\">→ bin "+bin+"</div>";
          pxEl.appendChild(div);
        }
      }

      var denom = Math.sqrt(H.reduce(function(s,v){return s+v*v;},0) + 1e-6);
      var Hn = H.map(function(v){ return v/denom; });

      // ---- Barras (coloridas por bin) ----
      var dados = modoNorm ? Hn : H;
      var maxD = Math.max.apply(null, dados.concat([0.001]));
      bars.innerHTML = "";
      dados.forEach(function(v, b){
        var col = document.createElement("div");
        col.style.cssText = "display:flex;flex-direction:column;align-items:center;gap:4px;";
        var barra = document.createElement("div");
        var altura = Math.round((v/maxD)*90) + 4;
        barra.style.cssText = "width:34px;height:"+altura+"px;background:"+CORES[b]+";border-radius:4px 4px 0 0;";
        var label = document.createElement("div");
        label.style.cssText = "font-family:monospace;font-size:10px;color:#4b5563;";
        label.textContent = modoNorm ? v.toFixed(4) : v.toFixed(2);
        var binLabel = document.createElement("div");
        binLabel.style.cssText = "font-size:9px;color:#9ca3af;";
        binLabel.textContent = "bin "+b;
        col.appendChild(barra);
        col.appendChild(label);
        col.appendChild(binLabel);
        bars.appendChild(col);
      });

      // ---- Passo a passo (floor + soma) ----
      var passos = [];
      for(var b=0;b<B;b++){
        var contribs = binsPorPixel.filter(function(p){ return p.bin===b; });
        var termos = contribs.map(function(p){ return p.mag.toFixed(2)+" (θ="+p.ang+"°→⌊"+(p.ang/largura).toFixed(2)+"⌋="+p.bin+")"; }).join(" + ");
        if(termos === "") termos = "(nenhum pixel)";
        passos.push("<span style=\\"color:"+CORES[b]+";font-weight:700;\\">H["+b+"]</span> = "+termos+" = <b>"+H[b].toFixed(2)+"</b>");
      }
      passosEl.innerHTML = passos.join("<br>");

      dbg.textContent = "H=[" + H.map(function(v){return v.toFixed(2);}).join(", ") + "]  |  Ĥ=[" +
        Hn.map(function(v){return v.toFixed(4);}).join(", ") + "]";
    }

    slEl.addEventListener("input", render);
    btnRaw.addEventListener("click", function(){ modoNorm = false; render(); });
    btnNorm.addEventListener("click", function(){ modoNorm = true; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0705");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.5:** EP07_05 Simulator: HOG Histogram of a Cell (mapping angles to bins)


<figure id="fig-07-sim-ep0705">
  <img src="imagens/fig-07-sim-ep0705.png" alt=" EP07_05 Simulator: HOG Histogram of a Cell (mapping angles to bins) " style="max-width:80%" />
  <figcaption><strong>Figura 7.5:</strong>  EP07_05 Simulator: HOG Histogram of a Cell (mapping angles to bins) </figcaption>
</figure>

In [ ]:
%%writefile EP07_05.py
# Python code

In [ ]:
TestSuite("EP07_05.py").run()

### EP07_06 🟣 Complete *Pipeline*: Descriptors + k-NN + Multi-Class Evaluation

This exercise integrates the three central stages of the chapter into a single *pipeline*, reproducing in miniature the **Practical Project 2** (classification of synthetic textures by LBP): a set of histograms of descriptors **already extracted** (as if they were LBP histograms) is used to train a k-NN classifier, which in turn is evaluated on an independent test set using a multi-class confusion matrix.

Unlike EP07_01, here the feature space has arbitrary dimension $H$ (the histogram size), there are more than two classes, and the distance metric is an input parameter — making it possible to reproduce the metric comparison experiment discussed in the chapter.

#### 📋 Implementation Guidelines

1. **Classes:** Read the integer $C$ (number of classes) followed by $C$ class names (*strings* without spaces), in the order in which they should appear in the confusion matrix.
2. **Configuration:** Read the integer $H$ (histogram dimension), the *string* $M$ (metric: `euclidiana` or `manhattan`) and the odd integer $k$.
3. **Training:** Read the integer $N$ and then $N$ lines, each containing the class name followed by $H$ real values (the descriptor histogram).
4. **Testing:** Read the integer $Q$ and then $Q$ lines, each containing the **actual** class name followed by $H$ real values (the descriptor histogram of the test sample).
5. **Distance:** For each test sample, calculate the distance to each training example using the metric $M$:
$$
d_{\text{euclidiana}}(u,v) = \sqrt{\sum_{j=1}^{H}(u_j-v_j)^2}, \qquad
d_{\text{manhattan}}(u,v) = \sum_{j=1}^{H} |u_j - v_j|.
$$
6. **k-NN Classification:** Select the $k$ closest training examples (distance tie broken by reading order, as in EP07_01) and classify by the majority class among them. In the case of a **voting tie** between two or more classes, choose the one that appears **first** in the class list from item 1.
7. **Confusion matrix:** Build a $C \times C$ matrix in which the row corresponds to the actual class and the column to the predicted class, following the class order from item 1.
8. **Accuracy:** Calculate the global accuracy as the ratio of correct predictions to $Q$.
9. **Output:** For each test sample, in input order, print the predicted class. Then print the confusion matrix (one row per actual class, values separated by spaces, in class order). Finally, print the accuracy rounded to 4 decimal places.

#### 📌 Computational Constraints

* **Selectable metric:** implement both distances; the metric $M$ defines which one is used throughout the execution (it is not possible to mix metrics in the same run).
* **Deterministic voting tie-break:** the criterion from item 6 (class list order) must be followed even when the tie involves more than two classes.
* **Training and test independence:** there is no need to validate that the test samples do not appear in training — assume the input is valid.

#### 🧠 Theoretical Foundation

| Exercise stage | Corresponding stage in the chapter |
|---|---|
| Training/test histograms already extracted | `descritor_lbp` applied to synthetic textures |
| Euclidean or Manhattan distance | `metric` parameter of the `KNeighborsClassifier` |
| Majority voting with $k$ neighbors | `KNeighborsClassifier.predict` |
| $C\times C$ confusion matrix | `confusion_matrix` from `scikit-learn` |
| Global accuracy | `accuracy_score` from `scikit-learn` |

This exercise highlights, in a controlled way, a result discussed in the chapter: the **choice of distance metric** and the **value of $k$** can change the predicted class for the same sample, even keeping the descriptor used fixed — reinforcing that, in classical pattern recognition, the descriptor, the metric, and the classifier form an interdependent system, not isolated parts.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: integer $C$ followed by $C$ class names.
* Line 2: integer $H$, *string* $M$, and integer $k$.
* Line 3: integer $N$.
* Next $N$ training lines: class name followed by $H$ real values.
* Next line: integer $Q$.
* Next $Q$ test lines: actual class name followed by $H$ real values.

**Output:**

* $Q$ lines with the predicted class of each test sample, in input order.
* $C$ lines with the confusion matrix (one row per actual class).
* Last line: `Acuracia: <value>`.

#### 📌 Examples

| Input (summarized) | Output | Observation |
|---|---|---|
| 2 granular listrada<br>2 euclidiana 1<br>4<br>granular 0.9 0.1<br>granular 0.8 0.2<br>listrada 0.1 0.9<br>listrada 0.2 0.8<br>2<br>granular 0.85 0.15<br>listrada 0.15 0.85 | granular<br>listrada<br>1 0<br>0 1<br>Acuracia: 1.0000 | With $k=1$, each test sample is classified by the closest training neighbor. |

> ### 📝 Nota
>
> This simulator uses a simplified set of **3 classes** (`granular`, `striped`, `spotted`) over fictional 2D points, solely to illustrate the voting, tie-breaking, and confusion matrix pipeline of k-NN. In **EP07_07**, you will apply this same logic to a real image mosaic, which introduces a fourth class (`checkered`) and replaces the 2D points with LBP histograms extracted directly from the image pixels.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0706" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0706 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0706 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0706 button:hover { background: #e8dfcf; }
  #sim-ep0706 button.sim-ep0706_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0706_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0706_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP07_06: Multi-Class k-NN Pipeline</span>
  <span class="sim-ep0706_pill">6 Train&middot; 3 Test&middot; 3 Classes</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Choose the metric, k value, and test sample (★). See the k nearest neighbors, the voting,
      the tie-breaking when necessary, and how this propagates to the confusion matrix and the accuracy of the whole set.
    </p>

    <!-- Controles -->
    <div style="display:flex;flex-wrap:wrap;gap:18px;justify-content:center;margin-bottom:16px;">
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Metric (M)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_be" class="ep0706_btn">Euclidean</button>
          <button id="ep0706_bm" class="ep0706_btn">Manhattan</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Neighbors (k)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_k1" class="ep0706_btn">k=1</button>
          <button id="ep0706_k3" class="ep0706_btn">k=3</button>
          <button id="ep0706_k5" class="ep0706_btn">k=5</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Test sample (★)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_t0" class="ep0706_btn">test 1</button>
          <button id="ep0706_t1" class="ep0706_btn">test 2</button>
          <button id="ep0706_t2" class="ep0706_btn">test 3</button>
        </div>
      </div>
    </div>

    <!-- Legenda -->
    <div id="ep0706_legenda" style="display:flex;gap:10px;justify-content:center;margin-bottom:10px;"></div>

    <!-- Dispersao 2D -->
    <div style="max-width:340px;margin:0 auto 16px auto;height:300px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa;">
      <div id="ep0706_svg_container" style="width:100%;height:100%;"></div>
    </div>

    <!-- Distancias ordenadas -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📏 Distances to the test sample (sorted) — <span style="font-weight:400;font-size:10px;color:#8a8672;">#i = reading order in the training list (hover)</span></div>
      <div id="ep0706_dists" style="display:grid;grid-template-columns:1fr 1fr;gap:2px 10px;font-family:monospace;font-size:10px;"></div>
    </div>

    <!-- Votacao -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🗳️ Voting among the k neighbors</div>
      <div id="ep0706_votos" style="display:flex;gap:10px;justify-content:center;margin-bottom:6px;"></div>
      <div id="ep0706_previsao" style="text-align:center;font-size:12px;font-weight:bold;"></div>
    </div>

    <!-- Matriz de confusao + acuracia (conjunto de teste inteiro) -->
    <div style="margin-bottom:8px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📋 Confusion matrix and accuracy — running the pipeline over the 3 test samples</div>
      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;justify-content:center;">
        <table id="ep0706_cm" style="border-collapse:collapse;font-size:11px;font-family:monospace;"></table>
        <div id="ep0706_acc" style="font-size:13px;font-weight:bold;color:#5e5a4a;"></div>
      </div>
    </div>

    <div id="ep0706_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0706 .ep0706_btn { font-size:11px;padding:5px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0706 .ep0706_btn.ativo { background:#7c3aed;color:#fff;border-color:#7c3aed; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var classes = ["granular","listrada","manchada"];
    var CORES = {granular:"#6366f1", listrada:"#f59e0b", manchada:"#10b981"};

    var trainPts = [
      {nome:"granular_1", cls:"granular", x:0.70, y:0.70},
      {nome:"granular_2", cls:"granular", x:0.25, y:0.85},
      {nome:"listrada_1", cls:"listrada", x:0.85, y:0.50},
      {nome:"listrada_2", cls:"listrada", x:0.60, y:0.15},
      {nome:"manchada_1", cls:"manchada", x:0.30, y:0.30},
      {nome:"manchada_2", cls:"manchada", x:0.15, y:0.55}
    ];
    var testPts = [
      {nome:"teste 1", cls:"granular", x:0.50, y:0.50},
      {nome:"teste 2", cls:"listrada", x:0.70, y:0.20},
      {nome:"teste 3", cls:"manchada", x:0.20, y:0.40}
    ];

    var svgContainer = root.querySelector("#ep0706_svg_container");
    var svg = svgNS("svg");
    svg.setAttribute("viewBox", "0 0 100 100");
    svg.setAttribute("style", "width:100%;height:100%;");
    svgContainer.appendChild(svg);
    var legendaEl = root.querySelector("#ep0706_legenda");
    var distsEl = root.querySelector("#ep0706_dists");
    var votosEl = root.querySelector("#ep0706_votos");
    var previsaoEl = root.querySelector("#ep0706_previsao");
    var cmEl = root.querySelector("#ep0706_cm");
    var accEl = root.querySelector("#ep0706_acc");
    var dbg = root.querySelector("#ep0706_debug");

    var be = root.querySelector("#ep0706_be"), bm = root.querySelector("#ep0706_bm");
    var bk1 = root.querySelector("#ep0706_k1"), bk3 = root.querySelector("#ep0706_k3"), bk5 = root.querySelector("#ep0706_k5");
    var bt0 = root.querySelector("#ep0706_t0"), bt1 = root.querySelector("#ep0706_t1"), bt2 = root.querySelector("#ep0706_t2");

    var metrica = "euclidiana", k = 1, testSel = 0;

    function dist(u, v){
      var dx = u.x-v.x, dy = u.y-v.y;
      if(metrica === "euclidiana") return Math.sqrt(dx*dx+dy*dy);
      return Math.abs(dx)+Math.abs(dy);
    }

    function knnPredict(xtest){
      var ds = trainPts.map(function(p, i){ return {i:i, p:p, d:dist(xtest, p)}; });
      ds.sort(function(a,b){ return a.d - b.d; }); // ordem estavel = desempate por ordem de leitura
      var viz = ds.slice(0, k);
      var votos = {}; classes.forEach(function(c){ votos[c]=0; });
      viz.forEach(function(v){ votos[v.p.cls]++; });
      var maxV = Math.max.apply(null, classes.map(function(c){return votos[c];}));
      var empatados = classes.filter(function(c){ return votos[c]===maxV; });
      var pred = empatados[0]; // primeira classe da lista entre as empatadas
      return {pred:pred, viz:viz, votos:votos, empatados:empatados, ordenados:ds};
    }

    function svgNS(tag){
      // Concatenado de propósito: evita que filtros de auto-link do Moodle
      // reconheçam "http://www.w3.org/2000/svg" como URL e insiram uma tag <a>
      // dentro desta string, o que quebraria a sintaxe do createElementNS.
      var SVG_NS = "http" + "://www.w3.org/2000/svg";
      return document.createElementNS(SVG_NS, tag);
    }

    function render(){
      be.classList.toggle("ativo", metrica==="euclidiana");
      bm.classList.toggle("ativo", metrica==="manhattan");
      bk1.classList.toggle("ativo", k===1);
      bk3.classList.toggle("ativo", k===3);
      bk5.classList.toggle("ativo", k===5);
      bt0.classList.toggle("ativo", testSel===0);
      bt1.classList.toggle("ativo", testSel===1);
      bt2.classList.toggle("ativo", testSel===2);

      // Legenda
      legendaEl.innerHTML = "";
      classes.forEach(function(c){
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:5px;font-size:11px;color:#374151;";
        chip.innerHTML = '<span style="width:10px;height:10px;border-radius:50%;background:'+CORES[c]+';display:inline-block;"></span>'+c;
        legendaEl.appendChild(chip);
      });

      var xt = testPts[testSel];
      var r = knnPredict(xt);
      var vizIdx = r.viz.map(function(v){ return v.i; });

      // ---- SVG: pontos de treino, linhas para vizinhos, estrela de teste ----
      svg.innerHTML = "";
      // grade leve
      for(var g=1; g<4; g++){
        var lineV = svgNS("line");
        lineV.setAttribute("x1", g*25); lineV.setAttribute("y1", 0);
        lineV.setAttribute("x2", g*25); lineV.setAttribute("y2", 100);
        lineV.setAttribute("stroke", "#eee"); lineV.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineV);
        var lineH = svgNS("line");
        lineH.setAttribute("x1", 0); lineH.setAttribute("y1", g*25);
        lineH.setAttribute("x2", 100); lineH.setAttribute("y2", g*25);
        lineH.setAttribute("stroke", "#eee"); lineH.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineH);
      }
      // linhas ate os vizinhos (desenhadas antes dos pontos, para ficarem por baixo)
      vizIdx.forEach(function(i){
        var p = trainPts[i];
        var line = svgNS("line");
        line.setAttribute("x1", xt.x*100); line.setAttribute("y1", (1-xt.y)*100);
        line.setAttribute("x2", p.x*100); line.setAttribute("y2", (1-p.y)*100);
        line.setAttribute("stroke", CORES[p.cls]); line.setAttribute("stroke-width", "0.6");
        line.setAttribute("stroke-dasharray", "1.5,1"); line.setAttribute("opacity", "0.7");
        svg.appendChild(line);
      });
      // pontos de treino
      trainPts.forEach(function(p, i){
        var isViz = vizIdx.indexOf(i) !== -1;
        if(isViz){
          var halo = svgNS("circle");
          halo.setAttribute("cx", p.x*100); halo.setAttribute("cy", (1-p.y)*100);
          halo.setAttribute("r", 5); halo.setAttribute("fill", "none");
          halo.setAttribute("stroke", CORES[p.cls]); halo.setAttribute("stroke-width", "0.8");
          svg.appendChild(halo);
        }
        var c = svgNS("circle");
        c.setAttribute("cx", p.x*100); c.setAttribute("cy", (1-p.y)*100);
        c.setAttribute("r", 3.2);
        c.setAttribute("fill", CORES[p.cls]);
        c.setAttribute("stroke", "#fff"); c.setAttribute("stroke-width", "0.6");
        c.setAttribute("opacity", isViz ? "1" : "0.55");
        svg.appendChild(c);
      });
      // estrela de teste
      var correto = (r.pred === xt.cls);
      var estCor = correto ? "#16a34a" : "#dc2626";
      var halo2 = svgNS("circle");
      halo2.setAttribute("cx", xt.x*100); halo2.setAttribute("cy", (1-xt.y)*100);
      halo2.setAttribute("r", 5.5); halo2.setAttribute("fill", "#fff");
      halo2.setAttribute("stroke", estCor); halo2.setAttribute("stroke-width", "0.8");
      svg.appendChild(halo2);
      var txt = svgNS("text");
      txt.setAttribute("x", xt.x*100); txt.setAttribute("y", (1-xt.y)*100+1.8);
      txt.setAttribute("text-anchor", "middle"); txt.setAttribute("font-size", "6.5");
      txt.setAttribute("fill", estCor);
      txt.textContent = "★";
      svg.appendChild(txt);

      // ---- Distancias ordenadas ----
      distsEl.innerHTML = "";
r.ordenados.forEach(function(v, ord){
  var dentroK = ord < k;
  var row = document.createElement("div");
  row.style.cssText = "display:flex;justify-content:space-between;align-items:center;padding:2px 6px;border-radius:6px;" +
    (dentroK ? "background:"+CORES[v.p.cls]+"22;border:1px solid "+CORES[v.p.cls]+";" : "background:#f9fafb;border:1px solid #f1f1f1;color:#9ca3af;");
  row.innerHTML =
    '<span style="display:flex;align-items:center;gap:4px;">' +
      (dentroK ? '✓' : '\u00A0') +
      '<span title="reading position in the original training list — used for tie-breaking when two distances are equal" ' +
        'style="background:#eee;color:#9ca3af;border-radius:3px;padding:0 3px;font-size:8.5px;cursor:help;">#' + (v.i+1) + '</span>' +
      ' ' + v.p.nome + ' <span style="color:'+CORES[v.p.cls]+';font-weight:700;">('+v.p.cls+')</span>' +
    '</span>' +
    '<span>d='+v.d.toFixed(4)+'</span>';
  distsEl.appendChild(row);
});

      // ---- Votacao ----
      votosEl.innerHTML = "";
      classes.forEach(function(c){
        var venceu = (c === r.pred);
        var empatou = r.empatados.length > 1 && r.empatados.indexOf(c) !== -1;
        var div = document.createElement("div");
        div.style.cssText = "text-align:center;border-radius:10px;padding:8px 14px;font-size:12px;" +
          (venceu ? "background:"+CORES[c]+"22;border:2px solid "+CORES[c]+";" : "background:#f9fafb;border:1px solid #e5e7eb;color:#9ca3af;");
        div.innerHTML = '<div style="font-weight:700;color:'+CORES[c]+';">'+c+'</div><div style="font-size:16px;font-weight:700;">'+r.votos[c]+'</div>' +
          (empatou ? '<div style="font-size:9px;color:#b91c1c;">empate</div>' : '');
        votosEl.appendChild(div);
      });
      var msgEmpate = r.empatados.length > 1 ? " (empate entre "+r.empatados.join(", ")+" — desempate pela ordem da lista de classes)" : "";
      previsaoEl.innerHTML = 'Classe prevista: <span style="color:'+CORES[r.pred]+';">'+r.pred+'</span>' + msgEmpate +
        ' &nbsp;|&nbsp; classe real: <span style="color:'+CORES[xt.cls]+';">'+xt.cls+'</span> ' + (correto ? '✅' : '❌');

      // ---- Matriz de confusao + acuracia sobre as 3 amostras de teste ----
      var cm = [[0,0,0],[0,0,0],[0,0,0]];
      var acertos = 0;
      var predsGlobais = [];
      testPts.forEach(function(tp){
        var rr = knnPredict(tp);
        predsGlobais.push(rr.pred);
        var iReal = classes.indexOf(tp.cls);
        var iPrev = classes.indexOf(rr.pred);
        cm[iReal][iPrev]++;
        if(rr.pred === tp.cls) acertos++;
      });
      var acc = acertos/testPts.length;

      var thead = '<tr><td></td>' + classes.map(function(c){ return '<td style="padding:4px 8px;color:'+CORES[c]+';font-weight:700;">'+c.slice(0,4)+'</td>'; }).join('') + '</tr>';
      var rows = classes.map(function(cReal, i){
        var cells = classes.map(function(cPrev, j){
          var v = cm[i][j];
          var diag = (i===j);
          var bg = v===0 ? '#fff' : (diag ? '#dcfce7' : '#fee2e2');
          return '<td style="padding:4px 10px;text-align:center;border:1px solid #e5e7eb;background:'+bg+';">'+v+'</td>';
        }).join('');
        return '<tr><td style="padding:4px 8px;color:'+CORES[cReal]+';font-weight:700;">'+cReal.slice(0,4)+'</td>'+cells+'</tr>';
      }).join('');
      cmEl.innerHTML = thead + rows;
      accEl.textContent = "Accuracy: " + acc.toFixed(4) + " (" + acertos + "/" + testPts.length + ")";

      dbg.textContent = "M="+metrica+" k="+k+" | teste_sel="+xt.nome+" | y_pred(todas)=["+predsGlobais.join(", ")+"]";
    }

    be.addEventListener("click", function(){ metrica="euclidiana"; render(); });
    bm.addEventListener("click", function(){ metrica="manhattan"; render(); });
    bk1.addEventListener("click", function(){ k=1; render(); });
    bk3.addEventListener("click", function(){ k=3; render(); });
    bk5.addEventListener("click", function(){ k=5; render(); });
    bt0.addEventListener("click", function(){ testSel=0; render(); });
    bt1.addEventListener("click", function(){ testSel=1; render(); });
    bt2.addEventListener("click", function(){ testSel=2; render(); });

    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0706");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.6:** Simulator EP07_06: k-NN Multi-Class *Pipeline* (voting, tie-breaking and confusion matrix)


<figure id="fig-07-sim-ep0706">
  <img src="imagens/fig-07-sim-ep0706.png" alt=" Simulator EP07_06: k-NN Multi-Class *Pipeline* (voting, tie-breaking and confusion matrix) " style="max-width:80%" />
  <figcaption><strong>Figura 7.6:</strong>  Simulator EP07_06: k-NN Multi-Class *Pipeline* (voting, tie-breaking and confusion matrix) </figcaption>
</figure>

In [ ]:
%%writefile EP07_06.py
# Python code

In [ ]:
TestSuite("EP07_06.py").run()

### EP07_07 ⚫ Real Classification of a Texture Mosaic via LBP + k-NN

In the previous exercises, the LBP descriptor (**EP07_04**) and the multiclass k-NN classifier (**EP07_06**) were studied separately, always based on data already provided as input — isolated $3\times3$ neighborhoods or previously extracted histograms. In this final exercise of the chapter, the program should **read a real image**, in **ASCII PGM (P2)** format, compute the LBP descriptor directly from the pixels, and then classify each region using k-NN, reproducing, on a reduced scale, the complete flow of a texture recognition system. This approach also anticipates the idea of **region mosaic classification**, related to semantic segmentation studied in a later chapter.

The interactive simulator from **EP07_06** used only three classes (`granular`, `striped`, and `blotchy`) represented by fictitious two-dimensional points. In this exercise, a fourth class, **checkered**, is added, and the points are replaced by LBP histograms extracted from a real image.

The input image is a **mosaic** formed by a $G\times G$ grid of square blocks of $S\times S$ pixels. Each block contains a sample of one of the four synthetic texture classes from the chapter: **granular**, **striped**, **blotchy**, or **checkered** (checkerboard pattern with alternating intensities). As in the other exercises in the book, image loading is performed by the didactic function `mm.readImg`.

> ### 💡 Why a single mosaic, rather than several images?
>
> The input gathers the $G \times G$ texture samples into a single **PGM** file, merely to simplify data reading and avoid opening multiple files. For the algorithm, this does not alter the processing: each block is treated independently, as if it were an isolated image.
> The only exception is the **border exclusion** (item 4 below).

#### 📋 Implementation Guidelines

1. **Reading image dimensions**

   Read, from standard input, two lines containing, respectively, the number of rows $L$ and the number of columns $C$ of the mosaic (both multiples of the block size $S$, with $L=C$).

2. **Image loading**

   Use the didactic function

   ```python
   f = mm.readImg(L, C)
   ```

   to read the $L \times C$ intensity values (grayscale, `uint8`) of the mosaic.

3. **Grid parameters**

   Read the integer $G$ (number of blocks per side) and the integer $S$ (side length of each block, in pixels), satisfying $L = C = G \times S$.

4. **LBP code computation per pixel**

   For each **interior** pixel of the image (i.e., one not on the global border of `f` — row or column $0$ or $L-1$/$C-1$), compute the LBP code with $P=8$ neighbors and radius $R=1$, traversing the neighbors **clockwise** starting from the upper-left corner, exactly as in EP07_04: `[lin-1][col-1]`, `[lin-1][col]`, `[lin-1][col+1]`, `[lin][col+1]`, `[lin+1][col+1]`, `[lin+1][col]`, `[lin+1][col-1]`, `[lin][col-1]`.

   Pixels on the global border of the image **do not** have a complete neighborhood and should be **ignored** (they do not contribute to any histogram). This includes border pixels that fall within the interior of a block (exclusion is always relative to the border of the entire image, not to the border of each block).

5. **Uniform LBP histogram per block (10 bins)**

   For each block $(i,j)$ of the grid ($i,j = 0,\ldots,G-1$), accumulate, among its valid pixels (item 4), a histogram $H^{(i,j)}$ of $10$ bins:

   * Considering the circular bit sequence $s_0,\ldots,s_7$ of the pixel (same transition rule as in EP07_04): if the number of transitions is $\le 2$ (**uniform** pattern), the pixel contributes to bin $\operatorname{popcount}(s_0,\ldots,s_7) \in \{0,\ldots,8\}$ (number of bits equal to `1`);
   * Otherwise (**non-uniform** pattern), the pixel contributes to bin $9$.

   At the end, normalize each block's histogram by dividing by the number of valid pixels it contains, obtaining $\hat H^{(i,j)}$, with $\sum_{b=0}^{9} \hat H^{(i,j)}[b] = 1$.

6. **Training prototypes**

   Read the integer $Ncl$ (number of classes) followed by $Ncl$ class names (the order defining the confusion matrix and voting tie-break, as in EP07_06); then read the *string* $M$ (metric: `euclidean` or `manhattan`) and the odd integer $k$; finally, read the integer $N$ (number of prototypes) and, for each one, the class name followed by $10$ real values (already normalized prototype histogram).

7. **k-NN classification of each block**

   For each block, compute the distance from $\hat H^{(i,j)}$ to each of the $N$ prototypes, using metric $M$ (same formulas as in EP07_06). Select the $k$ closest prototypes (distance tie-break by prototype reading order) and classify by majority class (voting tie-break by the class order from item 6).

8. **True labels and evaluation**

   Read, on a single line, the $G \times G$ **true** class names of each block, in row-major order of the grid (block $(0,0)$, $(0,1)$, …, $(0,G-1)$, $(1,0)$, …). Build the $Ncl \times Ncl$ confusion matrix (row = true class, column = predicted class) and the overall accuracy.

9. **Output**

   Print, for each block (in the same reading order of the true labels from item 8), the predicted class. Then print the confusion matrix (one row per true class, in the order of item 6). Finally, print the accuracy, rounded to 4 decimal places.

#### 📌 Computational Constraints

* **Fixed descriptor:** $P=8$, $R=1$, and $10$ bins (per item 5) are fixed in this exercise — they are not read from the input.
* **Global border exclusion, not per-block:** a pixel at the boundary between two blocks, but within the image interior, is valid and contributes normally to the histogram of the block to which it belongs.
* **Reading order as tie-break criterion:** both distance tie-breaking (item 7) and voting tie-breaking (item 7) follow exactly the same conventions as EP07_01 and EP07_06.
* **Prototypes as input, not learned:** unlike Practical Project 2, the training histograms are provided directly as input; the program must not generate synthetic textures.

#### 🧠 Theoretical Foundation

| Exercise step | Corresponding step in the chapter |
|---|---|
| Image reading via `mm.readImg` | Image acquisition in the pattern recognition *pipeline* |
| LBP code per pixel (EP07_04) | `local_binary_pattern(image, P=8, R=1, method="uniform")` |
| 10-bin histogram per block | `descritor_lbp` function from Practical Project 2 (`bins=10`, `range=(0, P+2)`) |
| k-NN classification with selectable metric (EP07_06) | `KNeighborsClassifier` trained on `X_textura` |
| $Ncl\times Ncl$ confusion matrix and accuracy | `confusion_matrix` and `accuracy_score` on `yt_teste` |

This exercise highlights, with real pixels rather than synthetic values, a limitation discussed in the final section of the chapter: texture classes visually distinct to a human observer — such as **granular** and **blotchy** — may produce similar LBP histograms when the considered neighborhood is small ($R=1$), since both exhibit a high frequency of non-uniform patterns at the scale of a single pixel. The **checkered** class, on the other hand, due to its regular and repetitive edges, tends to be separated more easily. The confusion matrix produced is expected to reflect precisely this pattern of partial confusion.

#### 📦 Input and Output Specification (VPL)

**Input:**

```
L
C
[L x C image matrix]
G S
Ncl class_name_1 ... class_name_Ncl
M k
N
class_name h0 h1 ... h9      (repeated N times)
label(0,0) label(0,1) ... label(G-1,G-1)
```

**Output:**

* $G \times G$ lines with the predicted class of each block, in grid reading order.
* $Ncl$ lines with the confusion matrix (one line per true class, values separated by spaces).
* Last line: `Accuracy: <value>`.

#### 📌 Example (manual verification)

To check the descriptor implementation before testing it on a complete mosaic, consider a $6\times6$ **homogeneous** image, with all pixels at intensity $100$, treated as a single block ($G=1$, $S=6$). Since every interior pixel has its 8 neighbors with intensity equal to the center ($g_p \ge g_c$ in all cases), all bits $s_p$ are `1`, the number of transitions is $0$ (uniform), and the bin is $\operatorname{popcount}(11111111)=8$. The histogram of the single block is therefore `0 0 0 0 0 0 0 0 1 0`.

| Input (summarized) | Output | Observation |
|---|---|---|
| 6<br>6<br>[36 values equal to 100]<br>1 6<br>2 uniform other<br>euclidean 1<br>2<br>uniform 0 0 0 0 0 0 0 0 1 0<br>other 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1<br>uniform | uniform<br>1 0<br>0 0<br>Accuracy: 1.0000 | Distance from the block to the `uniform` prototype is exactly $0$; the `other` class does not appear in the true label, so its row in the confusion matrix is null. |

#### 📌 Reference Files (.pgm)

For local debugging, two test mosaics in ASCII P2 format are provided (attached to this assignment; when integrating them into the chapter repository, save them in `all/cap07/data/EP07/`):

* 📥 **Case 1 — Simple mosaic (`Caso1_Mosaico_Simples.pgm`)**: $2\times2$ grid of $24\times24$-pixel blocks, one sample of each of the four classes, with low noise — useful for validating image reading and classification logic in a controlled scenario.
* 📥 **Case 2 — Mixed mosaic (`Caso2_Mosaico_Misto.pgm`)**: $3\times3$ grid of $16\times16$-pixel blocks, with repeated classes and greater variability — a scenario in which the confusion between **granular** and **blotchy** discussed in the Theoretical Foundation tends to manifest.

[Figura 7.7](#fig-07-ep07) displays both mosaics for visual inspection before implementation.

In [ ]:
import os
import urllib.request
import numpy as np

def garantir_e_baixar_arquivo(nome_arquivo):
    diretorio_local = "dados/EP07"
    caminho_local = os.path.join(diretorio_local, nome_arquivo)
    
    # Create the local directory if it does not exist
    if not os.path.exists(diretorio_local):
        os.makedirs(diretorio_local)
        
    # If the file does not exist locally, download from the remote repository
    if not os.path.exists(caminho_local):
        url_base = "https://raw.githubusercontent.com/fzampirolli/"
        url_base += "pdi-vc/master/all/cap07/dados/EP07"
        url_arquivo = f"{url_base}/{nome_arquivo}"
        print(f"Downloading {nome_arquivo} from GitHub...")
        try:
            urllib.request.urlretrieve(url_arquivo, caminho_local)
        except Exception as e:
            raise IOError(f"Erro ao baixar {nome_arquivo} do GitHub. ",
                          "Verifique a conexão ou a URL. Detalhes: {e}")
            
    return caminho_local

def ler_pgm_p2(caminho):
    with open(caminho) as f:
        linhas = [l for l in f.read().split() if l]
    assert linhas[0] == "P2"
    C, L = int(linhas[1]), int(linhas[2])
    maxv = int(linhas[3])
    valores = list(map(int, linhas[4:4 + L * C]))
    return np.array(valores, dtype=np.uint8).reshape(L, C)

# Ensures the download and obtains the correct path
arq_caso1 = garantir_e_baixar_arquivo("Caso1_Mosaico_Simples.pgm")
arq_caso2 = garantir_e_baixar_arquivo("Caso2_Mosaico_Misto.pgm")

# Reads the PGM matrices
caso1 = ler_pgm_p2(arq_caso1)
caso2 = ler_pgm_p2(arq_caso2)

mm.show(
    [caso1, caso2],
    titles=[
        "Case 1: Simple Mosaic\n(2x2 blocks, 1 sample/class)",
        "Case 2: Mixed Mosaic\n(3x3 blocks, repeated classes)",
    ],
    cols=2,
    figsize=(8, 4),
)

**Figura 7.7:** Reference mosaics (ASCII PGM format) used in EP07_07. Case 1: 2x2 grid with one sample of each class. Case 2: 3x3 grid with repeated classes and greater variability.


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0707" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">  
<style>
  #sim-ep0707 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0707 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #ede6d8; background: #f3efe6; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0707 button:hover { background: #e8e0cf; }
  #sim-ep0707 button.sim-ep0707_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0707_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #ede6d8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0707_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 EP07_07 Simulator: Mosaic Classification via LBP + k-NN</span>
  <span class="sim-ep0707_pill">⚫ full pipeline</span>
</div>


  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
3x3 mosaic of 12x12 blocks (L=C=36). LBP (P=8,R=1) computed pixel by pixel, with global border exclusion.      Adjust k and the metric and observe the classification of each block against 8 prototypes (2 per class).
   
   </p>
     
<div style="background:#fff3cd;border:1px solid #ffe69c;border-radius:8px;padding:8px 12px;margin-bottom:12px;font-size:11px;color:#7a5c00;">
  ⚠️ Synthetic textures generated by code, not the real .pgm files from EP07_07. Use this simulator to understand the algorithm flow, not as a difficulty reference among the classes.
</div>
     
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:24px;flex-wrap:wrap;align-items:center;">
      <div style="flex:1;min-width:180px;">
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
          <label style="font-size:12px;font-weight:bold;color:#2980b9;">k (number of neighbors)</label>
          <span id="ep0707_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span>
        </div>
<input id="ep0707_sl" style="width:100%;accent-color:#2980b9;" max="5" min="1" step="2" type="range" value="1">
      </div>
      <div>
        <label style="font-size:12px;font-weight:bold;color:#2980b9;display:block;margin-bottom:6px;">Metric</label>
        <select id="ep0707_metric" style="font-size:12px;padding:4px 8px;border-radius:6px;border:1px solid #ccc;">
          <option value="euclidiana">euclidean</option>
          <option value="manhattan">manhattan</option>
        </select>
      </div>
    </div>

    <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start;">
      <canvas id="ep0707_canvas" style="border-radius:8px;border:1px solid #ccc;"></canvas>
      <div id="ep0707_grid" style="flex:1;min-width:220px;display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <div id="ep0707_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;white-space:pre-line;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var S = 12, G = 3, L = G * S, SCALE = 5;   // bloco maior reduz o vazamento de borda; SCALE ajustado p/ manter o canvas ~180px
    var classesOrder = ["granular", "listrada", "manchada", "xadrez"];
    // Grade 3x3 com classes repetidas, análoga ao Caso 2 do enunciado
    var layout = [
      "granular", "listrada", "manchada",
      "xadrez",   "granular", "manchada",
      "listrada", "xadrez",   "granular"
    ];

    // --- Geração determinística de textura por pixel (didática, não os PGMs reais) ---
    function h(a, b, phase){
      var v = Math.sin((a + phase) * 12.9898 + (b + phase * 0.7) * 78.233 + phase * 3.1) * 43758.5453;
      return v - Math.floor(v);
    }
    function texturePixel(cls, r, c, phase){
      phase = phase || 0;
      switch(cls){
        case "granular": return h(r, c, phase) < 0.5 ? 220 : 30;
        case "listrada": return ((c + Math.floor(phase * 2)) % 4) < 2 ? 220 : 30;
        case "manchada": return h(Math.floor(r / 3), Math.floor(c / 3), phase) < 0.5 ? 200 : 60;
        case "xadrez":   return ((Math.floor(r / 2) + Math.floor(c / 2)) % 2 === 0) ? 230 : 20;
      }
    }

    function buildImage(){
      var img = [];
      for(var r = 0; r < L; r++){
        var row = [];
        for(var c = 0; c < L; c++){
          var bi = Math.floor(r / S), bj = Math.floor(c / S);
          row.push(texturePixel(layout[bi * G + bj], r, c, 0));
        }
        img.push(row);
      }
      return img;
    }

    // --- LBP: P=8, R=1, sentido horário, s_p = 1 se vizinho >= centro ---
    function lbpBin(patch, r, c){
      var center = patch[r][c];
      var neigh = [
        patch[r-1][c-1], patch[r-1][c], patch[r-1][c+1],
        patch[r][c+1],
        patch[r+1][c+1], patch[r+1][c], patch[r+1][c-1],
        patch[r][c-1]
      ];
      var bits = neigh.map(function(v){ return v >= center ? 1 : 0; });
      var trans = 0;
      for(var i = 0; i < 8; i++){ if(bits[i] !== bits[(i+1) % 8]) trans++; }
      if(trans <= 2) return bits.reduce(function(a,b){ return a+b; }, 0); // popcount 0..8
      return 9; // não uniforme
    }

    // Histograma de um patch isolado (usado para gerar protótipos), excluindo apenas a borda do patch
    function computeLBPHist(patch){
      var n = patch.length, m = patch[0].length;
      var hist = new Array(10).fill(0), count = 0;
      for(var r = 1; r < n - 1; r++){
        for(var c = 1; c < m - 1; c++){
          hist[lbpBin(patch, r, c)]++;
          count++;
        }
      }
      for(var k = 0; k < 10; k++) hist[k] = count > 0 ? hist[k] / count : 0;
      return hist;
    }

    // Histogramas por bloco da imagem completa, excluindo só a borda global (item 4/5 do enunciado)
    function computeMosaicHistograms(img){
      var hists = [], counts = [];
      for(var i = 0; i < G*G; i++){ hists.push(new Array(10).fill(0)); counts.push(0); }
      for(var r = 1; r < L - 1; r++){
        for(var c = 1; c < L - 1; c++){
          var bin = lbpBin(img, r, c);
          var idx = Math.floor(r/S) * G + Math.floor(c/S);
          hists[idx][bin]++;
          counts[idx]++;
        }
      }
      for(var b = 0; b < hists.length; b++){
        for(var k = 0; k < 10; k++) hists[b][k] = counts[b] > 0 ? hists[b][k] / counts[b] : 0;
      }
      return hists;
    }

    // --- Protótipos: 2 por classe (N=8), ordem de leitura fixa (usada no desempate) ---
    var prototypes = [];
    classesOrder.forEach(function(cls){
      [0, 5].forEach(function(phase){
        var Sp = S + 2, patch = [];
        for(var r = 0; r < Sp; r++){
          var row = [];
          for(var c = 0; c < Sp; c++) row.push(texturePixel(cls, r, c, phase));
          patch.push(row);
        }
        prototypes.push({ classe: cls, hist: computeLBPHist(patch) });
      });
    });

    function dist(u, v, metric){
      var s = 0;
      for(var i = 0; i < u.length; i++){
        s += metric === "euclidiana" ? (u[i]-v[i])*(u[i]-v[i]) : Math.abs(u[i]-v[i]);
      }
      return metric === "euclidiana" ? Math.sqrt(s) : s;
    }

    // Desempate de distância: ordem de leitura dos protótipos. Desempate de votação: ordem das classes.
    function classify(hist, k, metric){
      var cand = prototypes.map(function(p, idx){ return { classe: p.classe, d: dist(hist, p.hist, metric), idx: idx }; });
      cand.sort(function(a, b){ return a.d !== b.d ? a.d - b.d : a.idx - b.idx; });
      var viz = cand.slice(0, k);
      var votos = {};
      viz.forEach(function(v){ votos[v.classe] = (votos[v.classe] || 0) + 1; });
      var melhor = null, melhorN = -1;
      classesOrder.forEach(function(c){
        var n = votos[c] || 0;
        if(n > melhorN){ melhorN = n; melhor = c; }
      });
      return melhor;
    }

    var canvas = root.querySelector('#ep0707_canvas');
    canvas.width = L * SCALE; canvas.height = L * SCALE;
    var ctx = canvas.getContext('2d');
    var slK = root.querySelector('#ep0707_sl');
    var vlK = root.querySelector('#ep0707_vl');
    var selMetric = root.querySelector('#ep0707_metric');
    var gridEl = root.querySelector('#ep0707_grid');
    var dbg = root.querySelector('#ep0707_debug');

    var img = buildImage();
    var hists = computeMosaicHistograms(img);

    function render(){
      var k = parseInt(slK.value);
      var metric = selMetric.value;
      vlK.textContent = k;

      var preds = [];
      for(var idx = 0; idx < G*G; idx++) preds.push(classify(hists[idx], k, metric));

      var confusion = classesOrder.map(function(){ return new Array(classesOrder.length).fill(0); });
      var acertos = 0;
      for(var i2 = 0; i2 < G*G; i2++){
        var ri = classesOrder.indexOf(layout[i2]);
        var pi = classesOrder.indexOf(preds[i2]);
        confusion[ri][pi]++;
        if(layout[i2] === preds[i2]) acertos++;
      }
      var acc = acertos / (G*G);

      // Desenha a imagem real em tons de cinza
      for(var r = 0; r < L; r++){
        for(var c = 0; c < L; c++){
          var v = img[r][c];
          ctx.fillStyle = 'rgb(' + v + ',' + v + ',' + v + ')';
          ctx.fillRect(c*SCALE, r*SCALE, SCALE, SCALE);
        }
      }
      // Contorna cada bloco: verde = acerto, vermelho = erro
      for(var idx3 = 0; idx3 < G*G; idx3++){
        var bi = Math.floor(idx3 / G), bj = idx3 % G;
        ctx.strokeStyle = (preds[idx3] === layout[idx3]) ? '#10b981' : '#f43f5e';
        ctx.lineWidth = 2;
        ctx.strokeRect(bj*S*SCALE + 1, bi*S*SCALE + 1, S*SCALE - 2, S*SCALE - 2);
      }

      // Grade textual de apoio
      gridEl.innerHTML = '';
      for(var idx4 = 0; idx4 < G*G; idx4++){
        var ok = preds[idx4] === layout[idx4];
        var card = document.createElement('div');
        card.style.cssText = 'border-radius:8px;padding:6px;text-align:center;font-size:10px;border:2px solid ' + (ok ? '#10b981' : '#f43f5e') + ';';
        card.innerHTML = 'Real: ' + layout[idx4] + '<br><b style="color:' + (ok ? '#059669' : '#e11d48') + '">Pred: ' + preds[idx4] + (ok ? ' ✅' : ' ❌') + '</b>';
        gridEl.appendChild(card);
      }

      // Saída no mesmo formato do programa (itens 7-9 do enunciado)
      var linhas = [];
      linhas.push('Classes preditas (ordem de leitura da grade):');
      linhas.push(preds.join(' '));
      linhas.push('');
      linhas.push('Matriz de confusão (linhas=real, colunas=predita; ordem ' + classesOrder.join(',') + '):');
      confusion.forEach(function(lin){ linhas.push(lin.join(' ')); });
      linhas.push('');
      linhas.push('Acuracia: ' + acc.toFixed(4));
      dbg.textContent = linhas.join('\\n');
    }

    slK.addEventListener('input', render);
    selMetric.addEventListener('change', render);
    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0707');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.8:** Simulator EP07_07: Classification of a Texture Mosaic via LBP + k-NN


<figure id="fig-07-sim-ep0707">
  <img src="imagens/fig-07-sim-ep0707.png" alt=" Simulator EP07_07: Classification of a Texture Mosaic via LBP + k-NN " style="max-width:80%" />
  <figcaption><strong>Figura 7.8:</strong>  Simulator EP07_07: Classification of a Texture Mosaic via LBP + k-NN </figcaption>
</figure>

In [ ]:
%%writefile EP07_07.py
# Python code

In [ ]:
TestSuite("EP07_07.py").run()